In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text, _ = urllib.request.urlretrieve(url, "tinyshakespeare.txt")

with open("tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))


In [ ]:
import torch

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)
print(vocab_size)
print(data.shape)



In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
batch_size=32
block_size=8
def get_batch(data):
    ix=torch.randint(len(data)-block_size-1,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
import torch.nn as nn
class embedding(nn.Module):
        def __init__(self,vocab_size,d_model,block_size):
                super().__init__()
                self.token_embedding = nn.Embedding(vocab_size, d_model)
                self.position_embedding = nn.Embedding(block_size, d_model)
        def forward(self,x):
                B,T=x.shape
                tokens=self.token_embedding(x)
                position=torch.arrange(T)
                position=self.position_embedding(tokens)
                x=tokens+position
                return x

In [ ]:
import torch.nn.functional as F
class head(nn.Module):
    def __init__(self,d_model,head_size, block_size):
        super().__init__()
        self.query=nn.Linear(d_model,head_size,bias=False)
        self.key=nn.Linear(d_model,head_size,bias=False)
        self.value=nn.Linear(d_model,head_size,bias=False)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )
    def forward(self,x):
        B,T,C=x.shape
        q=self.query(x)
        k=self.key(x)
        v=self.value(x)
        weights=q @ k.transpose(-2,-1)
        weights=weights*(k.shape[-1]**0.5)
        weights = weights.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(weights, dim=-1)
        out=weights @ v
        return out
        


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, d_model, block_size):
        super().__init__()

        head_size = d_model // num_heads

        self.heads = nn.ModuleList([
            head(d_model, head_size, block_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):

        out = torch.cat([head(x) for head in self.heads], dim=-1)

        out = self.proj(out)

        return out

In [ ]:
class ff(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d_model,d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4,d_model),
        )
    def forward(self,x):
        return(self.net(x))

In [ ]:
class block(nn.Module):
    def __init__(self,d_model,block_size,num_head):
            super().__init__()
            self.ln1=nn.LayerNorm(d_model)
            self.atn=MultiHeadAttention(num_head,d_model,block_size)
            self.ln2=nn.LayerNorm(d_model)
            self.ffwd=ff(d_model)
    def forward(self,x):
          x=x+self.atn(self.ln1(x))
          x=x+self.ffwd(self.ln2(x))
          return x

In [ ]:
class GPTmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position_embedding=nn.Embedding(block_size,d_model)
        self.blocks = nn.Sequential(
            block(d_model, block_size, num_head),
            block(d_model, block_size, num_head),
            block(d_model, block_size, num_head),
        )
        self.ln_f=nn.LayerNorm(d_model)
        self.lm_f=nn.Linear(d_model,vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok = self.embedding(idx)
        pos = self.position_embedding(torch.arange(T, device=idx.device))

        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_f(x)

        if targets is None:
            return logits

        B, T, C = logits.shape

        loss = F.cross_entropy(
            logits.view(B * T, C),
            targets.view(B * T)
        )

        return logits, loss
        

            


In [ ]:
batch_size = 64      # was 32
block_size = 128     # was 8
d_model = 128        # was 32
max_iters = 10000 
vocab_size = len(chars)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPTmodel().to(device)



In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [ ]:

for step in range(max_iters):
    xb,yb=get_batch(train_data)
    xb = xb.to(device)
    yb = yb.to(device)
    logits,loss=model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 500 == 0:
        print(step, loss.item())

In [ ]:
def generate(idx,max_newtokens):
    for _ in range(max_newtokens):
        idx_co=idx[:,-block_size:]
        logits=model(idx_co)
        logits=logits[:,-1,:]
        prob=F.softmax(logits,dim=-1)
        next_token=torch.multinomial(prob,1)
        idx = torch.cat((idx, next_token), dim=1)

    return idx

In [ ]:
context = torch.zeros((1,1), dtype=torch.long,device=device)

generated = generate(context, 300)

print(decode(generated[0].tolist()))